# 构建文本嵌入模型

文本嵌入模型是许多强大的自然语言处理应用的基础，为增强文本生成模型等现有的强大技术奠定了基础。在本书中，我们已经将嵌入模型应用于多种场景，如监督分类、无监督分类、语义搜索等，甚至为ChatGPT等文本生成模型赋予记忆功能。

在自然语言处理领域，嵌入模型的重要性无论怎么强调都不为过，因为它们是众多应用背后的驱动力。因此，在本章中，我们将讨论多种构建和微调嵌入模型的方法，以提高其表征能力和语义能力。

我们首先了解一下什么是嵌入模型，以及它们通常是如何工作的。

## 10.1 嵌入模型

我们在很多章（第4章、第5章和第8章）中讨论过嵌入和嵌入模型，这充分说明了它们的实用性。在开始训练嵌入模型之前，我们先回顾一下嵌入模型的相关内容。

文本数据本身通常是非结构化的，很难处理。这些数据无法直接用于计算、可视化并生成可操作的结果。我们首先需要将文本数据转换成易于处理的形式，即数值表示。这个过程通常被称为嵌入(embedding)，即对输入进行嵌入处理，以输出可用的向量，如图10-1所示。

<div align="center">
    <img src="./picture/10-1.jpg">
    <p>图10-1：使用嵌入模型将文本输入（如文档、句子和短语）转换为数值表示，即嵌入</p>
</div>


对输入进行嵌入通常由LLM执行，我们称之为嵌入模型。嵌入模型的主要目标是尽可能准确地将文本数据表示为嵌入向量。

什么是准确的表示呢？通常，我们想要捕捉文档的语义本质，即其含义。如果我们能够捕捉到文档传达的核心内容，就有望把握文档的主旨。在实践中，这意味着我们希望看到相似文档的向量彼此接近，而内容完全不相关的文档，其向量应该不同。我们在本书中已多次提及语义相似性这一概念，如图10-2所示。该图是一个简化的例子。虽然二维可视化有助于说明嵌入向量的接近度(proximity)和相似性(similarity)，但这些嵌入向量通常存在于高维空间中。

<div align="center">
    <img src="./picture/10-2.jpg">
    <p>图10-2：语义相似性的理念是，我们期望具有相似含义的文本数据在n维空间中（这里展示了两个维度）彼此更接近</p>
</div>

然而，嵌入模型可以针对多种目的进行训练。例如，在构建情感分类器时，我们更关注文本的情感倾向而非语义相似性。如图10-3所示，我们可以微调模型，使文档在n维空间中的距离基于它们的情感倾向而非语义特性。

<div align="center">
    <img src="./picture/10-3.jpg">
    <p>图10-3：除了语义相似性，嵌入模型还可以被训练以关注情感倾向。在该图中，负面评价（红色）彼此接近，并与正面评价（绿色）明显不同</p>
</div>

无论如何，嵌入模型的目标都是学习使某些文档彼此相似的特征，而我们可以指导这个学习过程。通过向模型展示足够多的语义相似文档的示例，我们可以引导模型向语义分析的方向发展，而使用情感示例则会引导模型向情感分析的方向发展。

训练、微调和引导嵌入模型的方法很多，其中最强大且应用最广泛的技术是对比学习。

## 10.2 什么是对比学习

对比学习是训练和微调文本嵌入模型的一种主要技术。对比学习的目标是训练嵌入模型，使相似文档在向量空间中距离更近，而不相似文档相距更远。这看上去很熟悉，因为它与第2章中的word2vec方法非常相似。图10-2和图10-3也是对这一概念的运用。

对比学习的基本理念是，向模型输入相似的和不相似的文档对作为示例，这是学习文档之间的相似性或差异性并构建相关模型的最佳方式。为了准确地捕捉文档的语义特征，模型往往需要将一个文档与另一个文档进行对比，从而学习它们之间的相似之处或区别。这种对比过程非常有效，并且与文档写作的背景密切相关。图10-4展示了这一过程的整体框架。

对比学习还可以通过解释的本质来理解。一个很好的例子是一则趣闻。一名记者问一个劫匪：​“你为什么要抢银行？​”劫匪回答：​“因为银行里有钱。​”虽然这个答案在事实上是正确的，但记者的本意并不是问他为什么选择到银行抢劫，而是问他为什么要抢劫。这就叫作对比解释，指的是通过与其他选择进行对比来理解特定情况，即通过“为什么是P而不是Q”理解“为什么是P”。在这个例子中，记者的问题可以有多种解释方式，而提供一个替代选项或许是最好的建模方式：​“你为什么要抢银行(P)而不是遵守法律(Q)？​”

<div align="center">
    <img src="./picture/10-4.jpg">
    <p>图10-4：对比学习旨在教会嵌入模型判断文档之间的相似性。它通过向模型呈现具有不同程度相似性或差异性的文档组来实现这一目标</p>
</div>


替代选项对于理解问题非常重要，在嵌入通过对比学习来进行学习时，替代选项同样非常重要。我们通过向模型展示相似的和不相似的文档对，让模型开始学习使这些文档相似或不相似的特征，更重要的是，明白为什么会这样。

例如，你可以通过让模型发现“尾巴”​“鼻子”​“四条腿”等特征来教会它理解什么是狗。这个学习过程可能相当困难，因为特征通常没有明确的定义，而且可以从多个角度进行解释。一个有“尾巴”和“鼻子”的“四条腿”的生物也可能是猫。为了帮助模型朝着我们感兴趣的方向发展，我们实际上是在问它：​“为什么这是一只狗而不是一只猫？​”我们通过提供两个概念之间的对比，让模型开始学习定义这个概念的特征，以及哪些特征与之无关。当将问题构建为对比形式时，我们能获得更多信息。图10-5进一步说明了对比解释的概念。

<div align="center">
    <img src="./picture/10-5.jpg">
    <p>图10-5：当我们向嵌入模型输入不同的对比内容（不同程度的相似性）时，模型开始学习是什么让事物彼此不同，从而理解概念的独特特征</p>
</div>

在自然语言处理领域，一个最早且最流行的对比学习的例子是我们在第1章和第2章中讨论过的word2vec。该模型通过在句子中训练单个词来学习词的表示。在一个句子中，靠近目标词的词被构建成正例对，而随机采样的词被构建成负例对（不相似的对）​。换句话说，它通过将目标词的相邻词与非相邻词进行对比来训练模型。虽然并不广为人知，但这是自然语言处理领域利用神经网络进行对比学习的首批重大突破之一。

我们可以通过多种方式将对比学习应用于构建文本嵌入模型，最著名的技术和框架是sentence-transformers。

## 10.3 SBERT

尽管对比学习有多种形式，但在自然语言处理领域，推广这种技术的一个框架是sentence-transformers。该框架解决了原始BERT实现在创建句子嵌入时的一个主要问题，即计算开销。在sentence-transformers诞生前，句子嵌入通常使用交叉编码器(cross-encoder)架构，并结合BERT模型来实现。

交叉编码器允许两个句子同时通过Transformer网络进行处理，以预测两个句子的相似度。它通过在原始架构上添加分类头来实现这一点，该分类头可以输出相似度分数。然而，当你想在一个包含10000个句子的集合中找到相似度最高的配对时，计算量会迅速增加。这需要进行n*(n-1)/2=49995000次推理计算，因此会产生巨大的开销。此外，如图10-6所示，交叉编码器通常不会生成嵌入向量，而是输出输入句子之间的相似度分数。

<div align="center">
    <img src="./picture/10-6.jpg">
    <p>图10-6：交叉编码器的架构。两个句子被连接在一起，用<SEP>词元分隔，然后作为一个整体输入模型</p>
</div>

降低上述开销的一种方法是通过计算BERT模型的输出层的平均值或使用[CLS]词元来生成嵌入向量。然而，这种方法的效果已被证明比简单地计算词向量的平均值（如GloVe）还要差。

为解决这个问题，sentence-transformers的作者找到了一种方法来快速创建可以进行语义比较的嵌入向量，最终得到了一个优雅的方案来替代原始的交叉编码器架构。与交叉编码器不同，在sentence-transformers中，分类头被移除，取而代之的是在最终输出层上使用平均池化来生成嵌入向量。这个池化层通过对词嵌入取平均值，生成一个固定维度的输出向量，从而确保嵌入向量的大小固定。

sentence-transformers的训练使用孪生(siamese)架构。如图10-7所示，在这种架构中有两个完全相同的BERT模型，它们共享权重和神经网络架构。这两个模型接收句子作为输入，通过对词元嵌入进行池化来生成嵌入向量，然后根据句子嵌入的相似度进行优化。由于两个BERT模型的权重是相同的，因此我们可以使用单个模型，并依次输入句子。

<div align="center">
    <img src="./picture/10-7.jpg">
    <p>图10-7：原始sentence-transformers模型的架构，它采用了孪生网络（也称为双编码器）的结构</p>
</div>

这些句子对的优化是通过损失函数完成的，而损失函数的设计会对模型性能产生重大影响。在训练过程中，每个句子的嵌入向量与它们之间的差向量会被拼接在一起，构成一个新的表示，然后softmax分类器对得到的嵌入向量进行优化。

这种架构也被称为双编码器(bi-encoder)或SBERT(sentence-BERT)。虽然双编码器的运算速度相当快并能创建精准的句子表示，但交叉编码器通常比双编码器具有更好的性能，并且无须生成嵌入向量。

双编码器和交叉编码器都基于对比学习。通过优化句子对之间的相似性或差异，模型最终学习到句子的本质特征。

要实现对比学习，需要两个条件。首先，需要构成相似/不相似对的数据。其次，需要明确模型如何定义和优化相似性。

## 10.4 构建嵌入模型

构建嵌入模型的方法有很多，我们通常采用对比学习。这对许多嵌入模型来说至关重要，因为通过对比学习，模型能够高效地学习语义表示。

然而，这不是一个“零成本”的过程。我们需要理解如何生成对比样本、如何训练模型，以及如何正确评估模型。

### 10.4.1 生成对比样本

在预训练嵌入模型时，你会经常看到自然语言推理(NLI)数据集中的数据。NLI任务的目标是研究给定前提是否蕴含假设（蕴含）​、是否与假设矛盾（矛盾）或者两者都不成立（中性）​。

如图10-8所示，当前提是“He is in the cinemawatching Coco”​（他在电影院观看《寻梦环游记》​）​，而假设是“He is watching Frozen athome”​（他在家观看《冰雪奇缘》​）时，这些陈述是矛盾的；当前提是“He is in the cinemawatching Coco”​（他在电影院观看《寻梦环游记》​）​，而假设是“In the movie theater, he iswatching the Disney movie Coco”​（他在电影院观看迪士尼电影《寻梦环游记》​）时，这些陈述是蕴含的。

<div align="center">
    <img src="./picture/10-8.jpg">
    <p>图10-8：我们可以利用NLI数据集的结构来生成用于对比学习的负例（矛盾）和正例（蕴含）</p>
</div>

如果仔细观察“蕴含”和“矛盾”这两个概念，你会发现它们描述的是两个输入之间的相似度。因此，我们可以使用NLI数据集来生成对比学习所需的负例（矛盾）和正例（蕴含）​。

在构建和微调嵌入模型的过程中，我们将使用来自通用语言理解评估基准（General LanguageUnderstanding Evaluation benchmark，简称GLUE基准）的数据。GLUE基准包含9个语言理解任务，用于评估和分析模型性能。

GLUE基准的任务之一是多类型自然语言推理(MNLI)语料库，它包含392702个有推理关系标注（矛盾、中性、蕴含）的句子对。我们将使用其中的50000个标注句子对来创建一个最小示例，这样就不需要长时间的训练。不过请注意，数据集越小，训练或微调嵌入模型就越不稳定。如果可能，在保证数据质量的前提下，最好使用更大的数据集：

In [4]:
from datasets import load_dataset
# 从GLUE加载MNLI数据集
# 0 = 蕴含，1 = 中性，2 = 矛盾
train_dataset = load_dataset(
    "nyu-mll/glue", "mnli", split="train"
).select(range(50_000))
train_dataset = train_dataset.remove_columns("idx")

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

d:\study\python\AiAgent\图解大模型\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\admin\.cache\huggingface\hub\datasets--nyu-mll--glue. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\study\python\AiAgent\图解大模型\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_

mnli/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 52.2MB            

mnli/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mnli/validation_matched-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 1.21MB            

...

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating validation_matched split:   0%|          | 0/9815 [00:00<?, ? examples/s]

Generating validation_mismatched split:   0%|          | 0/9832 [00:00<?, ? examples/s]

Generating test_matched split:   0%|          | 0/9796 [00:00<?, ? examples/s]

Generating test_mismatched split:   0%|          | 0/9847 [00:00<?, ? examples/s]

我们来看一个示例：

In [5]:
train_dataset[2]

{'premise': 'One of our number will carry out your instructions minutely.',
 'hypothesis': 'A member of my team will execute your orders with immense precision.',
 'label': 0}

这展示了一个前提和假设之间存在蕴含关系的示例，因为它们是正相关的，且含义几乎相同。

### 10.4.2 训练模型

现在我们有了包含训练样本的数据集，接下来就可以创建嵌入模型了。通常我们会选择一个现有的sentence-transformers模型并进行微调，但在这个示例中，将从头开始训练一个嵌入模型。

这意味着我们必须明确两件事。首先，我们需要确定一个用于嵌入单词的预训练Transformer模型。我们将使用BERT基座模型（不区分大小写版）作为入门模型。当然，还有许多模型也经过了sentence-transformers的评估。最值得一提的是，当将microsoft/mpnet-base用作词嵌入模型时，通常能得到不错的结果：

In [9]:
from sentence_transformers import SentenceTransformer
# 使用一个基座模型
embedding_model = SentenceTransformer('../models/bert-base-uncased')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ../models/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


默认情况下，sentence-transformers中LLM的所有层都是可训练的。虽然可以冻结某些层，但通常不建议这样做，因为在所有层都解冻的情况下，模型性能往往更好。

其次，我们需要定义一个用于优化模型的损失函数。正如本节开始提到的，sentence-transformers最早使用的是softmax损失函数。为便于说明，我们先使用softmax，稍后再介绍性能更好的损失函数。

In [10]:
from sentence_transformers import losses
# 定义损失函数。在softmax损失函数中，我们还需要显式设置标签数量
train_loss = losses.SoftmaxLoss(
    model=embedding_model,
    sentence_embedding_dimension=embedding_model.get_sentence_embedding_dimension(),
    num_labels=3
)

C:\Users\admin\AppData\Local\Temp\ipykernel_31688\2068984750.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import losses
C:\Users\admin\AppData\Local\Temp\ipykernel_31688\2068984750.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  sentence_embedding_dimension=embedding_model.get_sentence_embedding_dimension(),
The `sentence_embedding_dimension` argument was renamed and is now deprecated. Please use `embedding_dimension` instead.


在训练模型之前，我们定义一个评估器来评估训练过程中模型的性能，并根据评估结果保存表现最佳的模型。

我们可以使用语义文本相似度基准(SemanticTextual Similarity Benchmark，STSB)来评估模型的性能。这是一个由人工标注的句子对构成的数据集，相似度得分在1和5之间。

我们使用这个数据集来探索模型在语义相似度任务上的表现。此外，我们还要处理STSB数据，确保所有的值都在0和1之间。

In [12]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
# 为STSB创建嵌入相似度评估器
val_sts = load_dataset("nyu-mll/glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine",
)

stsb/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  502kB            

stsb/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

stsb/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  151kB            

stsb/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

stsb/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  114kB            

stsb/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

现在我们有了评估器，接下来要创建SentenceTransformerTrainingArguments，这与使用Hugging Face Transformers进行训练类似（我们将在第11章中探讨）​。

In [13]:
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
# 定义训练参数
args = SentenceTransformerTrainingArguments(
    output_dir="base_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

C:\Users\admin\AppData\Local\Temp\ipykernel_31688\1622613997.py:1: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import SentenceTransformerTrainingArguments


需要注意以下参数。

num_train_epochs

训练轮次。为了加快训练速度，我们将其设为1，但通常建议把这个值设置得大一些。

per_device_train_batch_size

在训练过程中每个设备（如GPU或CPU）同时处理的样本数量。一般来说，该参数的值越大，训练速度越快。

per_device_eval_batch_size

在评估过程中每个设备（如GPU或CPU）同时处理的样本数量。一般来说，该参数的值越大，评估速度越快。

warmup_steps

学习率从0线性增加到初始学习率所需的步数。注意，在本次训练过程中，我们没有指定自定义的学习率。

fp16

启用此参数后，我们可以进行混合精度训练，使用16位浮点数(FP16)而不是默认的32位浮点数(FP32)进行计算。这可以减少内存使用量并有可能提高训练速度。

至此，我们已经定义了数据、嵌入模型、损失函数和评估器，可以开始训练模型了。我们可以使用SentenceTransformerTrainer来训练模型：

In [15]:
from sentence_transformers.trainer import SentenceTransformerTrainer
# 训练嵌入模型
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

C:\Users\admin\AppData\Local\Temp\ipykernel_31688\1513362456.py:1: DeprecationWarning: Importing from 'sentence_transformers.trainer' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.trainer' instead.
  from sentence_transformers.trainer import SentenceTransformerTrainer


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Column 'hypothesis' is at index 1, whereas a column with this name is usually expected at index 0. Note that the column order can be important for some losses, e.g. MultipleNegativesRankingLoss will always consider the first column as the anchor and the second as the positive, regardless of the dataset column names. Consider renaming the columns to match the expected order, e.g.:
dataset = dataset.select_columns(['hypothesis', 'entailment', 'contradiction'])


Step,Training Loss
100,1.070985
200,0.945012
300,0.881875
400,0.843946
500,0.821226
600,0.827970
700,0.804942
800,0.794613
900,0.779663
1000,0.773917


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1563, training_loss=0.813067255001837, metrics={'train_runtime': 134.6389, 'train_samples_per_second': 371.364, 'train_steps_per_second': 11.609, 'total_flos': 0.0, 'train_loss': 0.813067255001837, 'epoch': 1.0})

训练完模型后，我们可以使用评估器来获取这个任务的性能：

In [16]:
# 评估我们训练好的模型
evaluator(embedding_model)

{'pearson_cosine': 0.539524328518305, 'spearman_cosine': 0.6181355849449538}

在该过程中出现了几种不同的距离度量指标，我们最感兴趣的是pearson_cosine。它是中心化向量之间的余弦相似度，其值介于0和1之间，值越大表示相似度越高。我们得到的pearson_cosine的值为0.539，它将在本章中作为基准值。

### 10.4.3 深入评估

一个好的嵌入模型不仅仅能够在STSB测试中取得优异分数。正如上文所述，GLUE基准包含多个用于评估嵌入模型的任务。然而，用于评估嵌入模型的基准还有很多，为了统一评估过程，大规模文本嵌入基准(Massive Text EmbeddingBenchmark，MTEB)应运而生。MTEB涵盖8个嵌入任务，涉及58个数据集和112种语言。

为了公开比较前沿嵌入模型的性能，业界建立了MTEB排行榜，展示了各嵌入模型在相关任务上的得分。

In [24]:
import mteb

# 1. 获取任务对象
tasks = mteb.get_tasks(tasks=["Banking77Classification"])

# 2. 跑评估
results = mteb.evaluate(embedding_model, tasks=tasks)
print(results)

d:\study\python\AiAgent\图解大模型\.venv\Lib\site-packages\mteb\models\model_meta.py:966: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimensions = model.get_sentence_embedding_dimension()
d:\study\python\AiAgent\图解大模型\.venv\Lib\site-packages\mteb\models\model_meta.py:926: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embed_dim=model.get_sentence_embedding_dimension(),


Evaluating tasks:   0%|          | 0/1 [00:00<?, ?it/s]

The task 'Banking77Classification' is superseded by 'Banking77Classification.v2'. We recommend using the newer version of the task unless you are running a specific benchmark. See `get_task('Banking77Classification.v2').metadata.description` to get a description of the task and changes.
d:\study\python\AiAgent\图解大模型\.venv\Lib\site-packages\mteb\evaluate.py:262: UserWarning: The task 'Banking77Classification' is superseded by 'Banking77Classification.v2'. We recommend using the newer version of the task unless you are running a specific benchmark. See `get_task('Banking77Classification.v2').metadata.description` to get a description of the task and changes.
  task.check_if_dataset_is_superseded()


train.jsonl:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

d:\study\python\AiAgent\图解大模型\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\admin\.cache\huggingface\hub\datasets--mteb--banking77. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


test.jsonl:   0%|          | 0.00/365k [00:00<?, ?B/s]

model_name='SentenceTransformer' model_revision='no_revision_available' task_results=[TaskResult(task_name=Banking77Classification, main_score=0.61, scores=..., ...)] exceptions=[] experiment_name=None


In [29]:
import json
# pydantic v2
print(json.dumps(results.model_dump(mode="json"), indent=2, ensure_ascii=False))

{
  "model_name": "SentenceTransformer",
  "model_revision": "no_revision_available",
  "task_results": [
    {
      "dataset_revision": "0fd18e25b25c072e09e0d92ab615fda904d66300",
      "task_name": "Banking77Classification",
      "mteb_version": "2.22.1",
      "scores": {
        "test": [
          {
            "scores_per_experiment": [
              {
                "accuracy": 0.6042207792207792,
                "f1": 0.6038414419484831,
                "f1_weighted": 0.603841441948483,
                "precision": 0.6204227186994863,
                "precision_weighted": 0.6204227186994862,
                "recall": 0.6042207792207792,
                "recall_weighted": 0.6042207792207792,
                "ap": null,
                "ap_weighted": null
              },
              {
                "accuracy": 0.6077922077922078,
                "f1": 0.6067862649287956,
                "f1_weighted": 0.6067862649287956,
                "precision": 0.6191246272533306,
  

In [30]:
for tr in results.task_results:
    print("任务:", tr.task_name)
    print("主分数:", tr.main_score)
    print("详细:", tr.scores)
    print("---")

任务: Banking77Classification
主分数: 0.6134415584415585
详细: {'test': [{'scores_per_experiment': [{'accuracy': 0.6042207792207792, 'f1': 0.6038414419484831, 'f1_weighted': 0.603841441948483, 'precision': 0.6204227186994863, 'precision_weighted': 0.6204227186994862, 'recall': 0.6042207792207792, 'recall_weighted': 0.6042207792207792, 'ap': None, 'ap_weighted': None}, {'accuracy': 0.6077922077922078, 'f1': 0.6067862649287956, 'f1_weighted': 0.6067862649287956, 'precision': 0.6191246272533306, 'precision_weighted': 0.6191246272533304, 'recall': 0.6077922077922078, 'recall_weighted': 0.6077922077922078, 'ap': None, 'ap_weighted': None}, {'accuracy': 0.6152597402597403, 'f1': 0.6133470250068844, 'f1_weighted': 0.6133470250068843, 'precision': 0.6232603670435061, 'precision_weighted': 0.6232603670435061, 'recall': 0.6152597402597403, 'recall_weighted': 0.6152597402597403, 'ap': None, 'ap_weighted': None}, {'accuracy': 0.6275974025974026, 'f1': 0.6281925273617491, 'f1_weighted': 0.6281925273617492

这为我们提供了针对特定任务的多个评估指标，我们可以利用这些指标来探索模型的性能。

MTEB的优点不仅在于任务和语言的多样性，还在于可以节省评估时间。尽管有许多嵌入模型，但我们通常需要高准确率与低延迟的模型。语义搜索等使用嵌入模型的任务通常需要快速推理。

 由于在整个MTEB上测试模型可能需要几小时（取决于你的GPU）​，因此在本章中，我们将使用STSB来进行演示。

当你完成模型训练和评估后，重启Python环境很重要，这将为本章后续的训练示例清空显存。记得重启Python环境，确保所有显存都被清空。

### 10.4.4 损失函数

我们使用softmax损失函数来训练模型，为的是说明最早的sentence-transformers模型是如何训练的。实际上，可供选择的损失函数种类繁多，我们通常不建议使用softmax，因为其他损失函数可能更高效。

我们不会详细介绍每一个损失函数，这里只介绍两个常用且表现普遍较好的损失函数：

· 余弦相似度损失函数

· 多负例排序损失函数

除了这里讨论的损失函数，还有许多损失函数可供选择。例如，MarginMSE损失函数非常适合训练或微调交叉编码器。sentence-transformers框架实现了许多有趣的损失函数。

余弦相似度损失函数

余弦相似度损失函数是一个直观且易用的损失函数，适用于多种用例和数据集。它通常用于语义文本相似度任务。在这些任务中，我们为文本对分配相似度分数，并据此优化模型。

我们不会严格区分句子对是正例还是负例，而是假设句子对在一定程度上相似或不相似。通常，这个值介于0和1之间，分别表示不相似和相似（见图10-9）​。

余弦相似度损失函数的原理很简单——首先计算两段文本的两个嵌入向量之间的余弦相似度，然后将其与标注的相似度分数进行比较。模型将学会识别句子之间的相似度。

<div align="center">
    <img src="./picture/10-9.jpg">
    <p>图10-9：余弦相似度损失函数的目标是最小化语义相似的句子之间的余弦距离，并最大化语义不相似的句子之间的余弦距离</p>
</div>

余弦相似度损失函数最适合用于标注句子对数据，标注表示句子对之间的相似度，通常在0和1之间。要将余弦相似度损失函数用于NLI数据集，我们需要将蕴含(0)、中性(1)和矛盾(2)这3种标注转换为0和1之间的值。蕴含表示句子之间相似度高，所以我们将其相似度分数设为1。相比之下，中性和矛盾都表示相似度不高，所以我们将其相似度分数设为0。下面是使用余弦相似度损失函数的示例代码：

In [32]:
from datasets import Dataset, load_dataset
# 从GLUE加载MNLI数据集
# 0 = 蕴含, 1 = 中性, 2 = 矛盾
train_dataset = load_dataset(
    "nyu-mll/glue", "mnli", split="train"
).select(range(50_000))
train_dataset = train_dataset.remove_columns("idx")
# 中性/矛盾=0，蕴含=1
mapping = {2: 0, 1: 0, 0:1}
train_dataset = Dataset.from_dict({
    "sentence1": train_dataset["premise"],
    "sentence2": train_dataset["hypothesis"],
    "label": [float(mapping[label]) for label in train_dataset["label"]]
})

和之前一样，我们创建一个评估器：

In [33]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
# 为STSB创建嵌入相似度评估器
val_sts = load_dataset("nyu-mll/glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

然后，我们按照之前的步骤操作，但选择余弦相似度损失函数：

In [35]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
# 定义模型
embedding_model = SentenceTransformer("../models/bert-base-uncased")
# 损失函数
train_loss = losses.CosineSimilarityLoss(model=embedding_model)
# 定义训练参数
args = SentenceTransformerTrainingArguments(
    output_dir="cosineloss_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)
# 训练模型
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ../models/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
100,0.230393
200,0.169911
300,0.170638
400,0.158996
500,0.149995
600,0.156537
700,0.150152
800,0.157307
900,0.148240
1000,0.144048


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1563, training_loss=0.15658314473645776, metrics={'train_runtime': 132.8038, 'train_samples_per_second': 376.495, 'train_steps_per_second': 11.769, 'total_flos': 0.0, 'train_loss': 0.15658314473645776, 'epoch': 1.0})

训练后评估模型，得到以下分数：

In [36]:
# 评估我们训练的模型
evaluator(embedding_model)

{'pearson_cosine': 0.721203170468315, 'spearman_cosine': 0.7238975636263744}

pearson_cosine的值为0.72，与使用softmax损失函数的示例（pearson_cosine的值为0.539）相比有了很大的提升。这显示了损失函数对模型性能的影响。

请重启Python环境，下面我们将探索一个更常用且性能更好的损失函数，即多负例排序损失函数。

多负例排序损失函数

多负例排序损失函数也常被称为InfoNCE或NTXentLoss函数。它使用正例句子对或包含一对正例句子和一个不相关句子的三元组。这个不相关的句子被称为负例，它体现了正例句子与其他（不相关）内容之间的差异。

例如，可能存在问题/答案、图片/图片说明、论文标题/论文摘要等配对样本。这些配对样本的特点在于，我们可以确信它们是难正例(hard positive)对。在MNR损失函数中（见图10-10）​，负例对是通过将一个正例对中的元素与另一个正例对中的元素混合生成的。以论文标题和论文摘要为例，我们可以通过将一篇论文的标题与一个完全不相关的论文摘要组合起来生成负例对。这些负例被称为批内负例(in-batch negative)，也可以用于生成三元组。

<div align="center">
    <img src="./picture/10-10.jpg">
    <p>图10-10：MNR损失函数的目标是最小化相关文本对（如问题和相关答案）之间的距离，并最大化不相关文本对（如问题和不相关答案）之间的距离</p>
</div>

在生成这些正例对和负例对之后，我们计算它们的嵌入向量和余弦相似度。相似度分数用于回答一个问题：这些文本对是负例对还是正例对？换句话说，这被视为一个分类任务，我们可以使用交叉熵损失来优化模型。

要构建三元组，我们首先从一个锚句（标注为premise）开始，用于与其他句子进行比较。然后，使用MNLI数据集，我们仅选择正例句子对（标注为entailment）​。为了添加负例句子，我们随机采样句子作为hypothesis。

In [38]:
import random
from tqdm import tqdm
from datasets import Dataset, load_dataset
# 从GLUE加载MNLI数据集
mnli = load_dataset("nyu-mll/glue", "mnli", split="train").select(range(50_000))
mnli = mnli.remove_columns("idx")
mnli = mnli.filter(lambda x: True if x["label"] == 0 else False)
# 准备数据并添加软负例
train_dataset = {"anchor": [], "positive": [], "negative": []}
soft_negatives = list(mnli["hypothesis"])
random.shuffle(soft_negatives)
for row, soft_negative in tqdm(zip(mnli, soft_negatives)):
    train_dataset["anchor"].append(row["premise"])
    train_dataset["positive"].append(row["hypothesis"])
    train_dataset["negative"].append(soft_negative)
train_dataset = Dataset.from_dict(train_dataset)

16875it [00:00, 43434.16it/s]


由于我们只选择了标注为entailment的句子，因此行数从50000减少到了16875。

接下来，我们定义一个评估器：

In [39]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
# 为STSB创建嵌入相似度评估器
val_sts = load_dataset("nyu-mll/glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

然后，我们像之前一样进行训练，但使用MNR损失函数：

In [40]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
# 定义模型
embedding_model = SentenceTransformer("../models/bert-base-uncased")
# 损失函数
train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)
# 定义训练参数
args = SentenceTransformerTrainingArguments(
    output_dir="mnrloss_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)
# 训练模型
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ../models/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
100,0.321545
200,0.107729
300,0.080916
400,0.068108
500,0.071506


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=528, training_loss=0.12648728331833176, metrics={'train_runtime': 50.5087, 'train_samples_per_second': 334.101, 'train_steps_per_second': 10.454, 'total_flos': 0.0, 'train_loss': 0.12648728331833176, 'epoch': 1.0})

我们看看这个数据集和损失函数与之前的示例相比情况如何：

In [41]:
# 评估我们训练的模型
evaluator(embedding_model)

{'pearson_cosine': 0.8043654539735765, 'spearman_cosine': 0.80792866432176}

与我们之前使用余弦相似度损失函数训练的模型（pearson_cosine的值为0.72）相比，使用MNR损失函数训练的模型（pearson_cosine的值为0.80）似乎更加准确。

当使用MNR损失函数时，较大的批量大小通常表现更好，因为更大的批量会使任务更具挑战性。这是因为模型需要从更大的潜在句子对集合中找到最匹配的句子。你可以尝试设置不同的批量大小，以感受其效果。

我们使用MNR损失函数的方式存在一个缺点。由于负例是从其他问题/答案对中采样的，这些批内的或者说“简单”的负例可能与问题完全不相关，因此嵌入模型找到正确答案的任务变得相当容易。相反，我们希望有与问题非常相关但不是正确答案的负例。这样的负例被称为难负例(hard negative)。这会使嵌入模型的任务更具挑战性，因为它必须学习更细微的表示。因此，嵌入模型的性能通常会有显著提升。

下面是一个难负例的好例子。假设有这样一个问题：​“Howmany people live in Amsterdam?”​（阿姆斯特丹有多少人口？​）​，与这个问题相关的一个答案是：​“Almost a millionpeople live in Amsterdam”​（阿姆斯特丹的人口接近一百万）​。要生成一个好的难负例，理想情况下答案应包含与阿姆斯特丹和该城市人口数量有关的信息。例如，​“More than amillion people live in Utrecht, which is more thanAmsterdam”​（乌得勒支的人口超过一百万，比阿姆斯特丹还多）​。这个答案与问题相关，但不是真正的答案，所以这是一个好的难负例。图10-11展示了简单负例和难负例之间的差异。

<div align="center">
    <img src="./picture/10-11.jpg">
    <p>图10-11：简单负例通常与问题和答案都不相关。半难负例与问题和答案的主题有一些相似之处，但在某种程度上不相关。难负例与问题相关，但通常是错误答案</p>
</div>

收集负例大致可以分为以下三个步骤。

· 获取简单负例。像之前那样，通过随机采样文档来获取。

· 获取半难负例。我们可以使用预训练的嵌入模型，对所有句子嵌入应用余弦相似度，以找到高度相关的句子。通常，这并不会生成难负例，因为这种方法仅会找到相似的句子，而不是问题/答案对。

· 获取难负例。难负例通常需要手动标注（例如，通过生成半难负例的方式）​，或者使用生成模型来判断或生成句子对。

请重启Python环境，以便探索微调嵌入模型的不同方法。

## 10.5 微调嵌入模型

在10.4节中，我们介绍了从头开始训练嵌入模型的基础知识，并了解了如何利用损失函数来进一步优化其性能。这种方法虽然相当强大，但需要从头创建嵌入模型。这个过程可能相当耗费资源和时间。

相反，sentence-transformers框架允许几乎所有嵌入模型作为微调的基础。我们可以选择一个已经在大量数据上训练好的嵌入模型，并针对特定的数据或目的进行微调。

根据数据可用性和领域的不同，有多种微调模型的方法。我们将介绍其中的两种，并展示利用预训练嵌入模型的优势。

### 10.5.1 监督学习

微调嵌入模型最直接的方法是重复之前的模型训练过程，但要将bert-base-uncased替换为预训练的sentence-transformers模型。有很多模型可供选择，但通常来说，all-MiniLM-L6-v2在许多用例中表现良好，而且由于规模较小，其运行速度非常快。

我们使用与MNR损失函数示例中相同的数据来训练模型，但这次使用预训练的嵌入模型进行微调。和之前一样，我们先加载数据并创建评估器：

In [42]:
from datasets import load_dataset
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
# 从GLUE加载MNLI数据集
# 0 = 蕴含, 1 = 中性, 2 = 矛盾
train_dataset = load_dataset(
    "nyu-mll/glue", "mnli", split="train"
).select(range(50_000))
train_dataset = train_dataset.remove_columns("idx")
# 为STSB创建嵌入相似度评估器
val_sts = load_dataset("nyu-mll/glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

训练步骤与之前示例中的训练步骤类似，但我们可以使用预训练的嵌入模型来替代bert-base-uncased：

In [44]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
# 定义模型
embedding_model = SentenceTransformer('../models/all-MiniLM-L6-v2')
# 损失函数
train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)
# 定义训练参数
args = SentenceTransformerTrainingArguments(
    output_dir="finetuned_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)
# 训练模型
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Column 'hypothesis' is at index 1, whereas a column with this name is usually expected at index 0. Note that the column order can be important for some losses, e.g. MultipleNegativesRankingLoss will always consider the first column as the anchor and the second as the positive, regardless of the dataset column names. Consider renaming the columns to match the expected order, e.g.:
dataset = dataset.select_columns(['hypothesis', 'entailment', 'contradiction'])


Step,Training Loss
100,0.156400
200,0.111361
300,0.118007
400,0.114414
500,0.112098
600,0.097779
700,0.116340
800,0.101211
900,0.107242
1000,0.108050


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1563, training_loss=0.10966996908645484, metrics={'train_runtime': 64.7848, 'train_samples_per_second': 771.786, 'train_steps_per_second': 24.126, 'total_flos': 0.0, 'train_loss': 0.10966996908645484, 'epoch': 1.0})

对这个模型进行评估，我们得到以下分数：

In [45]:
# 评估我们训练的模型
evaluator(embedding_model)

{'pearson_cosine': 0.8479696808052222, 'spearman_cosine': 0.8480665496725888}

0.85的分数是我们目前看到的最高分，但别忘了我们用于微调的预训练模型已经在完整的MNLI数据集上进行了训练，而我们只使用了50000个样本。这可能看起来有些多余，但这个示例演示了如何在自己的数据上微调预训练的嵌入模型。

除了使用预训练的BERT模型（如bert-base-uncased）或可能的领域外模型（如all-mpnet-base-v2）​，你也可以在预训练的BERT模型上执行掩码语言建模，将其适配到自己的目标领域，然后将这个经过微调的BERT模型作为基础训练嵌入模型。这是一种领域适配方法。在第11章中，我们将在预训练模型上应用掩码语言建模。

请注意，训练或微调模型的主要难点在于找到合适的数据。对于这些模型，我们不仅需要庞大的数据集，数据本身的质量也必须很高。开发正例对通常比较直接，但增加难负例对会显著加大创建高质量数据的难度。

和之前一样，请重启Python环境，为接下来的示例释放显存。

### 10.5.2 增强型SBERT

训练或微调这些嵌入模型的一个缺点是它们通常需要大量的训练数据。许多这类模型是用超过十亿个句子对训练的。对于我们的用例而言，提取如此多的句子对通常是不可能的，因为在许多情况下，只有几千个标注数据点可用。

幸运的是，有一种方法可以增强数据，使得我们在只有少量标注数据的情况下也能微调嵌入模型。这个过程被称为增强型SBERT。

在这个过程中，我们的目标是增强少量的标注数据，使其可用于常规训练。增强型SBERT利用速度较慢但更精准的交叉编码器架构(BERT)来增强和标注更大的输入对集合。这些新标注的数据对随后被用于微调双编码器(SBERT)。

如图10-12所示，增强型SBERT包含以下步骤：

步骤1，使用小型标注数据集（黄金数据集）微调交叉编码器(BERT)；

步骤2，创建新的句子对；

步骤3，使用微调后的交叉编码器标注新的句子对（白银数据集）​；

步骤4，在扩展数据集（黄金数据集+白银数据集）上训练双编码器(SBERT)。

这里，黄金数据集是一个规模较小但完全标注的数据集，包含真实标注。白银数据集也是完全标注的，但不一定是真实标注，因为它是通过交叉编码器的预测生成的。

<div align="center">
    <img src="./picture/10-12.jpg">
    <p>图10-12：增强型SBERT的工作原理是：先在小型黄金数据集上训练交叉编码器，然后用它来标注未标注数据集以生成更大的白银数据集。最后，同时使用黄金数据集和白银数据集来训练双编码器</p>
</div>

在执行上述步骤之前，我们先准备数据。在原始的50000个文档中选取10000个文档组成小数据集，以模拟只有有限标注数据的情况。就像在余弦相似度损失函数的示例中那样，我们将蕴含的相似度分数设为1，而将中性和矛盾的相似度分数设为0。

In [46]:
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset, Dataset
from sentence_transformers import InputExample
from sentence_transformers.datasets import NoDuplicatesDataLoader
# 为交叉编码器准备具有10 000个文档的小数据集
dataset = load_dataset("nyu-mll/glue", "mnli", split="train").select(range(10_000))
mapping = {2: 0, 1: 0, 0:1}
# 数据加载器
gold_examples = [
    InputExample(texts=[row["premise"], row["hypothesis"]], label=mapping[row["label"]])
    for row in tqdm(dataset)
]
gold_dataloader = NoDuplicatesDataLoader(gold_examples, batch_size=32)
# 使用pandas DataFrame，以更方便地处理数据
gold = pd.DataFrame(
    {
    "sentence1": dataset["premise"],
    "sentence2": dataset["hypothesis"],
    "label": [mapping[label] for label in dataset["label"]]
    }
)

C:\Users\admin\AppData\Local\Temp\ipykernel_31688\467417000.py:5: DeprecationWarning: Importing from 'sentence_transformers.datasets' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.datasets' instead.
  from sentence_transformers.datasets import NoDuplicatesDataLoader
100%|██████████| 10000/10000 [00:00<00:00, 69963.02it/s]


这就是黄金数据集，因为它是有标注的，且代表了我们的真实标注。

使用这个黄金数据集训练交叉编码器（步骤1）​。

In [47]:
from sentence_transformers.cross_encoder import CrossEncoder
# 在黄金数据集上训练交叉编码器
cross_encoder = CrossEncoder("../models/bert-base-uncased", num_labels=2)
cross_encoder.fit(
    train_dataloader=gold_dataloader,
    epochs=1,
    show_progress_bar=True,
    warmup_steps=100,
    use_amp=False
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: ../models/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss


在训练完交叉编码器后，我们使用剩余的40000个句子对（来自包含50000个句子对的原始数据集）作为白银数据集（步骤2）​。

In [48]:
# 通过使用交叉编码器预测标注来准备白银数据集
silver = load_dataset(
    "nyu-mll/glue", "mnli", split="train"
).select(range(10_000, 50_000))
pairs = list(zip(silver["premise"], silver["hypothesis"]))

如果你没有额外的未标注句子对，也可以从原始的黄金数据集中随机采样。例如，你可以从一行中取premise，从另一行中取hypothesis来创建新的句子对。这样你就可以轻松地生成10倍于原始数据的句子对，并用交叉编码器对其进行标注。

然而，这种策略可能生成明显更多的不相似的句子对而非相似的句子对。作为替代方案，我们可以使用预训练的嵌入模型来嵌入所有候选句子对，并通过语义搜索为每个输入句子检索排名前k的句子。这种粗略的重排序过程让我们能够关注那些可能更相似的句子对。尽管由于预训练的嵌入模型并未在我们的数据上训练过，这些句子的选择仍然基于近似，但这种方法比随机采样要好得多。

请注意，在这个示例中，我们假设这些句子对是未标注的。我们将使用经过微调的交叉编码器来标注这些句子对（步骤3）​。

In [49]:
import numpy as np
# 使用经过微调的交叉编码器标注句子对
output = cross_encoder.predict(
    pairs, apply_softmax=True,
show_progress_bar=True
)
silver = pd.DataFrame(
    {
        "sentence1": silver["premise"],
        "sentence2": silver["hypothesis"],
        "label": np.argmax(output, axis=1)
    }
)

Batches:   0%|          | 0/1250 [00:00<?, ?it/s]

现在我们有了黄金数据集和白银数据集，只需要将它们组合，然后像之前那样训练我们的嵌入模型：

In [50]:
# 组合黄金数据集和白银数据集
data = pd.concat([gold, silver], ignore_index=True, axis=0)
data = data.drop_duplicates(subset=["sentence1", "sentence2"], keep="first")
train_dataset = Dataset.from_pandas(data, preserve_index=False)

和之前一样，我们需要定义一个评估器：

In [51]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
# 为STSB创建嵌入相似度评估器
val_sts = load_dataset("nyu-mll/glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

我们像之前一样训练模型，只是现在使用增强后的数据集：

In [52]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
# 定义模型
embedding_model = SentenceTransformer("../models/bert-base-uncased")
# 损失函数
train_loss = losses.CosineSimilarityLoss(model=embedding_model)
# 定义训练参数
args = SentenceTransformerTrainingArguments(
    output_dir="augmented_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)
# 训练模型
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ../models/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
100,0.216984
200,0.158756
300,0.143121
400,0.139182
500,0.136678
600,0.133019
700,0.133741
800,0.132266
900,0.130424
1000,0.130947


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1563, training_loss=0.13913588545220218, metrics={'train_runtime': 151.1644, 'train_samples_per_second': 330.752, 'train_steps_per_second': 10.34, 'total_flos': 0.0, 'train_loss': 0.13913588545220218, 'epoch': 1.0})

最后，我们对模型进行评估：

In [53]:
evaluator(embedding_model)

{'pearson_cosine': 0.7078545577666293, 'spearman_cosine': 0.7173779149989813}

余弦相似度损失函数示例的原始版本在使用完整数据集时得分为0.70，而我们仅使用其中20%的数据，就获得了0.71的分数！

这种方法能够扩展现有数据集的规模，而无须人工标注成千上万的句子对。你可以通过仅在黄金数据集上训练嵌入模型来测试白银数据集的质量。性能差异反映了白银数据集在提升模型质量方面具有多大的潜在作用。

请再次重启Python环境，我们将进入最后一个示例，即无监督学习。

## 10.6 无监督学习

要创建嵌入模型，我们通常需要标注数据。然而，并非所有现实世界的数据集都带可用的标注集。因此，我们转而寻找无须预定义标注就能训练模型的技术——无监督学习。目前存在许多方法，如SimCSE（Simple Contrastive Learning ofSentence Embeddings，句子嵌入的简单对比学习）、CT（Contrastive Tension，对比张力）、TSDAE（Transformer-based Sequential Denoising Auto-Encoder，基于Transformer的序列去噪自编码器）和GPL（Generative Pseudo-Labeling，生成式伪标签）。

在本节中，我们将重点介绍TSDAE，因为它在无监督任务和领域适配方面都表现出色。

### 10.6.1 TSDAE

TSDAE是一种非常优雅的通过无监督学习创建嵌入模型的方法。该方法假设我们完全没有标注数据，也不要求我们人为创建标签。

TSDAE的基本思想是通过删除输入句子中一定比例的词来为其添加噪声。这个“受损”的句子被输入编码器中，编码器的上方有一个池化层，将其映射为句子嵌入。基于这个句子嵌入，解码器尝试重建原始句子，但不包含人为添加的噪声。这里的核心概念是：句子嵌入越准确，重建的句子就越准确。

这种方法与掩码语言建模非常相似。在掩码语言建模中，我们试图重建和学习某些被掩码的词。这里，我们不是重建被掩码的词，而是尝试重建整个句子。

训练完成后，我们可以使用编码器从文本生成嵌入向量，因为解码器仅用于判断嵌入向量是否能准确地重建原始句子（见图10-13）​。

<div align="center">
    <img src="./picture/10-13.jpg">
    <p>图10-13：TSDAE随机删除输入句子中的词，然后将这个句子传入编码器生成句子嵌入，再基于这个句子嵌入重建原始句子</p>
</div>

由于我们只需要一组不带任何标注的句子，因此训练这个模型非常简单直接。首先，下载一个外部分词器，用于去噪过程：

In [59]:
# 下载额外的分词器
import nltk
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

然后，从数据中创建普通的句子，并移除所有标签以模拟无监督的设置：

In [60]:
from tqdm import tqdm
from datasets import Dataset, load_dataset
from sentence_transformers.datasets import DenoisingAutoEncoderDataset

# 加载 MNLI
mnli = load_dataset("nyu-mll/glue", "mnli", split="train").select(range(25_000))

# 转成 list 再拼接
flat_sentences = list(mnli["premise"]) + list(mnli["hypothesis"])

# 去重
flat_sentences = list(set(flat_sentences))

# 加噪声
damaged_data = DenoisingAutoEncoderDataset(flat_sentences)

# 组装
train_dataset = {"damaged_sentence": [], "original_sentence": []}
for data in tqdm(damaged_data):
    train_dataset["damaged_sentence"].append(data.texts[0])
    train_dataset["original_sentence"].append(data.texts[1])

train_dataset = Dataset.from_dict(train_dataset)
print(train_dataset)

100%|██████████| 48353/48353 [00:03<00:00, 15683.44it/s]

Dataset({
    features: ['damaged_sentence', 'original_sentence'],
    num_rows: 48353
})


这将创建一个包含50000个句子的数据集。当我们检查数据时，可以发现第一个句子是损坏的句子，第二个句子是原始句子。

In [61]:
train_dataset[0]

{'damaged_sentence': 'just the try the voting yeah',
 'original_sentence': 'well just the just the opportunity to try the voting machines yeah'}

第一个句子展示了带噪声的数据，而第二个句子展示了原始句子。创建数据后，我们像之前一样定义一个评估器：

In [62]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
# 为STSB创建嵌入相似度评估器
val_sts = load_dataset("nyu-mll/glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

接下来，我们像之前一样进行训练，但使用[CLS]词元作为池化策略，而不是对词元嵌入进行平均池化。在关于TSDAE的论文中，这被证明更有效，因为平均池化会丢失位置信息，而使用[CLS]词元则不会。

In [63]:
from sentence_transformers import models, SentenceTransformer
# 创建嵌入模型
word_embedding_model = models.Transformer("../models/bert-base-uncased")
pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), "cls")
embedding_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ../models/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\admin\AppData\Local\Temp\ipykernel_31688\2424228730.py:4: FutureWarning: The `get_word_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimens

使用我们的句子对，我们需要一个损失函数来尝试使用噪声句子重构原始句子，这个损失函数即DenoisingAutoEncoderLoss（去噪自编码器损失函数）​。这样，模型将学习如何准确地表示数据。这类似于掩码操作，但不需要知道实际掩码的位置。

此外，我们绑定了两个模型的参数。编码器的嵌入层和解码器的输出层不使用单独的权重，而是共享权重。这意味着一个层的权重更新也会反映在另一个层中。

In [64]:
from sentence_transformers import losses
# 使用去噪自编码器损失函数
train_loss = losses.DenoisingAutoEncoderLoss(
    embedding_model, tie_encoder_decoder=True
)
train_loss.decoder = train_loss.decoder.to("cuda")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertLMHeadModel LOAD REPORT from: ../models/bert-base-uncased
Key                                                                | Status     | 
-------------------------------------------------------------------+------------+-
bert.pooler.dense.bias                                             | UNEXPECTED | 
bert.pooler.dense.weight                                           | UNEXPECTED | 
cls.seq_relationship.bias                                          | UNEXPECTED | 
cls.seq_relationship.weight                                        | UNEXPECTED | 
bert.encoder.layer.{0...11}.crossattention.output.dense.weight     | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.output.LayerNorm.weight | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.self.key.bias           | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.self.query.weight       | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.output.dense.bias       | MISSING    | 
bert.encod

最后，与之前一样训练模型，但我们减小了批量大小，因为使用这个损失函数会增加内存使用量：

In [65]:
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
# 定义训练参数
args = SentenceTransformerTrainingArguments(
    output_dir="tsdae_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)
# 训练模型
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
100,6.966481
200,4.889529
300,4.633422
400,4.503759
500,4.377859
600,4.290274
700,4.264594
800,4.146795
900,4.026419
1000,4.084106


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3023, training_loss=4.03172588947787, metrics={'train_runtime': 290.2356, 'train_samples_per_second': 166.599, 'train_steps_per_second': 10.416, 'total_flos': 0.0, 'train_loss': 4.03172588947787, 'epoch': 1.0})

训练完成后，我们评估模型，以探索这种无监督技术的表现如何：

In [66]:
# 评估训练好的模型
evaluator(embedding_model)

{'pearson_cosine': 0.7321205325911346, 'spearman_cosine': 0.7402152439265256}

在拟合模型后，我们获得了0.73的分数。考虑到我们是使用无标注数据完成所有训练的，这个分数着实令人赞叹。

### 10.6.2 使用TSDAE进行领域适配

当我们只有很少或完全没有标注数据时，通常使用无监督学习的方法来创建文本嵌入模型。然而，无监督学习技术的表现通常不如监督学习技术，而且难以学习特定领域的概念。

这时领域适配(domain adaptation)就派上用场了。它的目标是将现有的嵌入模型更新到一个包含不同于源领域主题的特定文本领域。图10-14展示了不同领域在内容上的差异。目标领域（域外）通常包含未在源领域（域内）中出现的词语和主题。

<div align="center">
    <img src="./picture/10-14.jpg">
    <p>图10-14：领域适配的目标是创建一个嵌入模型，并使其从一个领域泛化到另一个领域</p>
</div>

领域适配的一种方法称为自适应预训练。首先，使用无监督学习技术（如前面讨论的TSDAE或掩码语言建模）对特定领域的语料库进行预训练。然后，如图10-15所示，使用域内或域外的训练数据集对该模型进行微调。虽然目标领域的数据是首选，但由于我们从目标领域的无监督训练开始，域外数据也同样有效。

<div align="center">
    <img src="./picture/10-15.jpg">
    <p>图10-15：领域适配可以通过自适应预训练和自适应微调来实现</p>
</div>

运用本章所学的所有知识，你应该能够复现这个流程。首先，可以使用TSDAE在目标领域训练嵌入模型，然后使用常规监督训练或增强版SBERT进行微调。

## 10.7 小结

在本章中，我们探讨了通过多种任务创建和微调嵌入模型的方法。我们首先讨论了嵌入的概念及其在将文本数据表示为数值格式中的作用，然后探索了许多嵌入模型的基础技术，即对比学习，它主要从文档的相似/不相似对中学习。

基于流行的嵌入框架sentence-transformers，我们使用预训练的BERT模型创建了嵌入模型，并探索了多种损失函数，如余弦相似度损失函数和MNR损失函数。我们还讨论了收集文档的相似/不相似对或三元组对最终模型性能的重要性。

随后，我们研究了微调嵌入模型的技术。我们讨论了监督学习技术和无监督学习技术，如用于领域适配的增强版SBERT和TSDAE。与创建嵌入模型相比，微调通常需要较少的数据，是将现有嵌入模型适配到所需领域的一个很好的方法。

在第11章中，我们将讨论微调用于分类的表示模型的方法，涉及BERT模型和嵌入模型，还将介绍各种微调技术。